In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import legogp as lego
import objax
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import batchjax

In [3]:
# generate data
P = 10

x = np.linspace(0, 1, 100)
y = np.sin(x*10)

X = x[:, None]
Y = y[:, None]

print(X.shape, Y.shape)

(100, 1) (100, 1)


In [4]:
YP = np.hstack([Y+p for p in range(P)])

In [5]:
# Construct model

m_prior_list = objax.ModuleList([
    lego.models.GP(X)
    for p in range(P)
])

m_post_list = objax.ModuleList([
    lego.models.GP(X, Y+p)
    for p in range(P)
])

multi_output_gp = lego.models.GP(X, YP)

/Users/ohamelijnck/Documents/projects/legogp/legogp/models/gp_prior.py:27: UserWarning: Using default ARD RBF kernel with input dim 1 
  warnings.warn(f'Using default ARD RBF kernel with input dim {self.input_dim} ')
/Users/ohamelijnck/Documents/projects/legogp/legogp/models/batch_gp.py:82: UserWarning: Using default ARD RBF kernel
  warnings.warn('Using default ARD RBF kernel')
/Users/ohamelijnck/Documents/projects/legogp/legogp/models/batch_gp.py:107: UserWarning: Using default Gaussian likelihood
  warnings.warn('Using default Gaussian likelihood')


In [6]:
mu_arr, var_arr = multi_output_gp.predict_f(X, diagonal=True)

In [10]:
multi_output_gp.covar(X, X).shape

NameError: name 'num_returned_args' is not defined

In [52]:
multi_output_gp.get_objective()

DeviceArray(1375.6632, dtype=float32)

In [53]:
np.sum([m_post_list[p].get_objective() for p in range(P)])

1375.6632

In [44]:
def get_objective(y_p, m, m2):
    return jnp.sum(y_p), [m.get_objective(), 1]

In [46]:
%%timeit -r 1 -n 1
# Manual 
a_list = []
b_list = []
for p in range(P):
    a, b = get_objective(YP[:, p], m_post_list[p], m_post_list[p])
    a_list.append(a)
    b_list.append(b)
print(np.array(a_list))
print(np.array(b_list))

[ 17.919315 117.91932  217.9193   317.9193   417.9193   517.9193
 617.9193   717.9194   817.9193   917.9194  ]
[[116.6503067    1.        ]
 [118.19216919   1.        ]
 [120.89883423   1.        ]
 [124.77030182   1.        ]
 [129.80657959   1.        ]
 [136.00764465   1.        ]
 [143.37353516   1.        ]
 [151.90420532   1.        ]
 [161.59970093   1.        ]
 [172.45999146   1.        ]]
136 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [50]:
%%timeit -r 1 -n 1
# vmap approach
a_val, b_val = batchjax.batch_fn(
    get_objective, 
    [YP, m_post_list, m_post_list], 
    axes=[1, 0, 0], 
    out_dim=2
)
print(a_val)
print(b_val)

[[ 17.919315 117.91932  217.91931  317.9193   417.9193   517.9193
  617.9193   717.9194   817.9194   917.9194  ]]
[[116.6503     1.      ]
 [118.19217    1.      ]
 [120.89884    1.      ]
 [124.770294   1.      ]
 [129.80656    1.      ]
 [136.00766    1.      ]
 [143.37352    1.      ]
 [151.9042     1.      ]
 [161.5997     1.      ]
 [172.45999    1.      ]]
73.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [48]:
%%timeit -r 1 -n 1
a_val, b_val = batchjax.batch_or_loop(
    get_objective, 
    [YP, m_post_list, m_post_list], 
    axes=[1, 0, 0], 
    dim = P,
    out_dim=2,
    batch_flag=False
)
print(a_val)
print(b_val)

[ 17.919315 117.91932  217.9193   317.9193   417.9193   517.9193
 617.9193   717.9194   817.9193   917.9194  ]
[[116.65031    1.      ]
 [118.19217    1.      ]
 [120.898834   1.      ]
 [124.7703     1.      ]
 [129.80658    1.      ]
 [136.00764    1.      ]
 [143.37354    1.      ]
 [151.9042     1.      ]
 [161.5997     1.      ]
 [172.45999    1.      ]]
138 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [20]:
np.sum(X)

50.0